# Multi-Model Training

Train and compare multiple model definitions from `poke_agent/model_catalog.py`.

- **Neural** entries train sequentially and save separate checkpoints.
- **Heuristic** entries are instant agents (random, first_legal, ...).
- **Hybrid** entries combine a neural policy with a heuristic fallback.

This notebook does **not** replace `poke_agent_training.ipynb` and will not interrupt an active run there.

## 1. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists() and (ROOT.parent / "requirements.txt").exists():
    ROOT = ROOT.parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from poke_agent.paths import print_runtime_info

print_runtime_info(ROOT)
print("torch", torch.__version__)

## 2. Model catalog

Edit `poke_agent/model_catalog.py` to add models, set `ACTIVE_MODEL`, and choose `TRAIN_MODELS`.

In [ ]:
from poke_agent.model_catalog import ACTIVE_MODEL, MODEL_CATALOG, TRAIN_MODELS
from poke_agent.model_registry import describe_catalog, validate_catalog

validate_catalog()
print("ACTIVE_MODEL:", ACTIVE_MODEL)
print("TRAIN_MODELS:", TRAIN_MODELS)
describe_catalog()

## 3. Shared dataset

In [ ]:
from poke_agent.config import build_config
from poke_agent.dataset import prepare_training_tensors
from poke_agent.device import torch_device

BASE_CONFIG = build_config(ROOT)
DEVICE = torch_device()
TENSORS = prepare_training_tensors(BASE_CONFIG, DEVICE)
print("rows", TENSORS.x.shape[0], "features", TENSORS.x.shape[1])

## 4. Train selected neural models

In [ ]:
from poke_agent.multi_train import train_catalog_models

REPORTS = train_catalog_models(
    root=ROOT,
    tensors=TENSORS,
    device=DEVICE,
    base_config=BASE_CONFIG,
)
REPORTS

## 5. Inspect heuristic / hybrid agents

In [ ]:
from poke_agent.agents import resolve_agent
from poke_agent.simulator import load_simulator

sim = load_simulator(ROOT)
if not sim.available or sim.to_observation_class is None:
    print("cg-lib unavailable — agent wiring only, no live CABT play")
else:
    for model_id, spec in MODEL_CATALOG.items():
        if spec["kind"] == "heuristic":
            agent = resolve_agent(model_id, to_observation_class=sim.to_observation_class)
            print(model_id, type(agent).__name__, getattr(agent, "strategy", ""))
        elif spec["kind"] == "hybrid":
            agent = resolve_agent(model_id, to_observation_class=sim.to_observation_class, neural_agent=None)
            print(model_id, type(agent).__name__, agent.mode)